## Task 1: Multi-Object Based Synopsis
- background modeling
- object detection
- euclidean distance tracker
- 

In [6]:
# task 1: multi-object base synopsis

# imports:
import cv2
import numpy as np
#background modeling:

frame_count = 0

# function for extracting frames from the videos and returns the grey frames in an array
def extract_frames(video_path):
  '''takes the filepath of the video and extracts its frames. Returns an array of grey frames '''
  cap = cv2.VideoCapture(video_path)

  # Check if video opened successfully
  if not cap.isOpened():
    print("Error opening video file")

  # get video details for output
  frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = cap.get(cv2.CAP_PROP_FPS)
  frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
  print(f"frame size: {frame_width}x{frame_height}")
  print(f"fps:{fps}")
  print(f"frame count:{frame_count}")

  count = 0
  frames = [] # initialize array
  # open the video and loop thru the frames to extract them
  print("looping thru frames to extract:")
  while(cap.isOpened()):
    # print("inside loop")
    ret, frame = cap.read() # read the frame
    if ret == True:
      print(f"extracting frame number {count}")
      frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) # make sure only one grey channel instead of rgb
      frames.append(frame) # save the frame into list
      count += 1
    else:
          print("End of video or error occurred.")
          break
  cap.release()
  return frames

# function to get BG0 given the frames
def get_BG0(frames):
  ''' takes the extracted frames of the video and calculates the first bakground model BG0.'''
  # get first two frames
  frame0 = frames[0]
  frame1 = frames[1]
  print(frame0.shape)
  rows, cols = frame0.shape
  BG0 = np.zeros((rows,cols), dtype = np.uint8) # create empty BG model
  # calculate average of both frame and put into BG0:
  BG0 = ((frame0.astype(np.uint16)+ frame1.astype(np.uint16))//2).astype(np.uint8)

  return BG0


# FGit = It ? |BGit+1 - It| > T : 0 OR FGit = It ? |BGit - It| > T : 0
# function to calculate FG from curr frame and next BG OR curr BG
def foreground(It, BGt_1):
  '''takes the curr frame and the curr background model. Returns the foreground (FG) and the mask(Msk)'''
  rows, cols = It.shape
  FG = np.zeros((rows,cols), dtype = np.uint8) # create empty foreground
  Msk = np.zeros((rows,cols), dtype = np.uint8) # create empty Mask
  thresh = 50 # set threhsold to 50
  # for i in range(rows):
  #   for j in range(cols):
  #     diff = BGt_1[i,j].astype(np.int16) - It[i,j].astype(np.int16)
  #     diff = np.abs(diff) # get abs diff betw next background and curr frame
  #     if diff > thresh: # check diff with threshold
  #       FG[i,j] = It[i,j]
  #       Msk[i,j] = 255
  #     else:
  #       FG[i,j] = 0
  #       Msk[i,j] = 0
  diff = np.abs(BGt_1.astype(np.int16)-It.astype(np.int16))
  FG = np.where((diff >thresh), It, 0)
  Msk = np.where((diff > thresh),255,0)
  return FG, Msk

# function to update background model -  need to use Msk instead of beta
def background_beta(It, BGt, beta):
  '''takes the curr frame, the curr background model, and beta. Calculates next background model '''
  rows, cols = It.shape
  BGt_1 = np.zeros((rows,cols), dtype = np.uint8) # create empty next BG model
  # for i in range(rows):
  #   for j in range(cols):
  #     w_sum = (1-beta) * BGt[i,j].astype(np.int16) + beta * It[i,j].astype(np.int16) # calculate weighted sum using beta
  #     BGt_1[i,j] = round(w_sum) # convert to int using roundign and put into next BG model
  w_sum = (1- beta) * BGt.astype(np.float32) + beta * It.astype(np.float32)
  BGt_1 = np.round(w_sum).astype(np.uint8)
  return BGt_1

# alternative, uses the background model creator that doesn't use masks
# function to create the foreground images, the corresponding mask images, and updates the background model over all frames
def moving_average_BG_alternate(frames, BGt, beta, path):
  ''' takes the frames, beta, and BG0 (BGt) to get the foreground, mask, and updated background models for each frame'''

  c = len(frames)
  for i in range(1,c): # loop over frames starting from second frame
    frame = frames[i] # get the curr frame
    # BGt_1 = background(frame, BGt, beta) # next background model
    # CHECK THIS!!!!!
    # FG, Msk = foreground(frame, BGt_1) # foreground using next background
    FG, Msk = foreground(frame, BGt) # foreground using curr background
    BGt_1 = background_beta(frame, BGt, beta) # next background model


    # write as images
    cv2.imwrite(f"{path}/Background/background_alternate_{i:04}.png",BGt_1)
    cv2.imwrite(f"{path}/Foreground/foreground_alternate_{i:04}.png",FG)
    cv2.imwrite(f"{path}/Mask/mask_alternate_{i:04}.png",Msk)
    BGt = BGt_1.copy() # set curr BG model for next iteration as the next background model from this iteration

def writeVideo(output_path, frames_path, fps=30):
    ''' used to create output videos from the frames'''  
    # create codec for writing the video
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    # create output for writing the grey video
    cap = cv2.VideoCapture(frames_path) # sequence of images

    if not cap.isOpened():
        print(f"Error: Could not open image sequence at {frames_path}")
        return

    # 2. Get dimensions automatically from the first image frame
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"frame size: {frame_width}x{frame_height}")
    print(f"fps:{fps}")
    print(f"frame count:{frame_count}")

    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height), isColor=False)


    count = 0
    # loop through the image sequence and write into the out vid
    print("looping through image sequence to write vid")
    while(cap.isOpened()):
        # print("inside writing video loop")
        ret, frame = cap.read()
        if ret == True:
            out.write(frame)
            print(f"writing frame number {count}")
            count += 1
        else:
                print("End of video or error occurred.")
                break

    cap.release()
    out.release()

beta = 0.4

# test on video
frames1 = extract_frames('tabletennis_short.mp4')

BG0_1 = get_BG0(frames1) # prev - BGt
cv2.imwrite(f"Results/Background/background_alternate_0000.png",BG0_1)

# using only beta for background model
moving_average_BG_alternate(frames1, BG0_1, beta, 'Results' )
writeVideo('tabletennis_foregrounds_alternate.mp4', 'Results/Foreground/foreground_alternate_%04d.png')
writeVideo('tabletennis_backgrounds_alternate.mp4', 'Results/Background/background_alternate_%04d.png')
writeVideo('tabletennis_masks_alternate.mp4', 'Results/Mask/mask_alternate_%04d.png')
print("saved videos")



# object detection


# euclidean distance tracker

frame size: 1920x1080
fps:30.0
frame count:1330.0
looping thru frames to extract:
extracting frame number 0
extracting frame number 1
extracting frame number 2
extracting frame number 3
extracting frame number 4
extracting frame number 5
extracting frame number 6
extracting frame number 7
extracting frame number 8
extracting frame number 9
extracting frame number 10
extracting frame number 11
extracting frame number 12
extracting frame number 13
extracting frame number 14
extracting frame number 15
extracting frame number 16
extracting frame number 17
extracting frame number 18
extracting frame number 19
extracting frame number 20
extracting frame number 21
extracting frame number 22
extracting frame number 23
extracting frame number 24
extracting frame number 25
extracting frame number 26
extracting frame number 27
extracting frame number 28
extracting frame number 29
extracting frame number 30
extracting frame number 31
extracting frame number 32
extracting frame number 33
extracting

## Task 2: Extend Pipeline with Video Object Segmentation with Memory (SAM2)

In [ ]:
# task 2: extend pipeline with SAM2